# BERTurk Duygu Analizi — Colab Eğitimi

Bu defter `bert_train.py`'yi Colab GPU'sunda çalıştırıp eğitilmiş checkpoint'i
Hugging Face Hub'a yükler.

**Önce yapılması gerekenler:** yerel değişiklikleri GitHub'a push et. Bu defter
repoyu klonluyor, dolayısıyla push etmediğin düzeltmeler burada olmaz.

**Runtime → Change runtime type → T4 GPU** seçtiğinden emin ol.

Beklenen süre: 3 epoch, yaklaşık 2-3 saat. Yorumlar kısa (medyan 29 token) ve
dinamik padding devrede, bu yüzden `max_length: 256` değerinin ima ettiğinden
çok daha hızlı ilerler.

## 1. GPU kontrolü

GPU yoksa devam etme — CPU'da bu eğitim günler sürer.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'GPU yok! Runtime > Change runtime type > T4 GPU sec.'
print('GPU:', torch.cuda.get_device_name(0))
print('torch:', torch.__version__)

## 2. Checkpoint'leri Drive'a bağla (önerilir)

Colab oturumu koparsa yerel disk silinir. Checkpoint'ler Drive'da durursa
`bert_train.py` yeniden çalıştırıldığında kaldığı yerden devam eder.

Bedeli: her kayıt ~1.3 GB (model + optimizer state) ve Drive yazması yavaştır.
Hızı önemsiyorsan bu hücreyi atla ve `save_steps` değerini `bert_hparams.yaml`
içinde yükselt — ama o zaman kopma riskini kabul ediyorsun.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_CKPT = '/content/drive/MyDrive/duyguanalizi_ckpt'
os.makedirs(DRIVE_CKPT, exist_ok=True)
print('checkpoint hedefi:', DRIVE_CKPT)

## 3. Repoyu klonla

In [ ]:
%cd /content
![ -d duyguanalizi ] || git clone https://github.com/bariskiratt/duyguanalizi.git
%cd /content/duyguanalizi
!git pull

# Checkpoint klasorunu Drive'a yonlendir (2. hucre calistirildiysa)
import os
os.makedirs('artifacts', exist_ok=True)
if 'DRIVE_CKPT' in globals() and not os.path.exists('artifacts/bert_mlp_ckpt'):
    os.symlink(DRIVE_CKPT, 'artifacts/bert_mlp_ckpt')
    print('artifacts/bert_mlp_ckpt -> Drive')
!ls -la artifacts/

## 4. Bağımlılıklar

`requirements.txt` içindeki `torch` pin'i **kasten kurulmuyor** — Colab'ın
önyüklü CUDA'lı torch'u zaten var, üzerine yazmak 2.5 GB indirir ve CUDA
uyumsuzluğu riski taşır. Kırılmanın yaşandığı yer transformers, onu pinliyoruz.

In [ ]:
!pip install -q \
    transformers==5.14.1 \
    tokenizers==0.22.2 \
    safetensors==0.8.0 \
    accelerate==1.14.0 \
    datasets==5.0.0 \
    scikit-learn==1.9.0 \
    PyYAML==6.0.3 rich tqdm

import transformers, accelerate, datasets
print('transformers', transformers.__version__)
print('accelerate  ', accelerate.__version__)
print('datasets    ', datasets.__version__)

## 5. Veri kontrolü

Eğitimi başlatmadan önce verinin gerçekten beklendiği gibi olduğunu gör.

In [ ]:
from datasets import load_from_disk
from collections import Counter

for split in ['bert_train', 'bert_val']:
    ds = load_from_disk(f'data/processed/{split}')
    print(f'{split}: {len(ds):,} satir | {dict(Counter(ds["labels"]))}')

## 6. Batch boyutu — muhtemelen dokunma

Mevcut ayar `batch_size: 16` + `gradient_accumulation_steps: 2` (etkin batch 32).

Sezgi "batch'i büyüt, GPU'yu daha iyi doyur" der ama **ölçüm bunu desteklemiyor.**
M4 Pro üzerinde gerçek veriyle ölçüldüğünde `batch_size: 32` örnek başına daha
*yavaş* çıktı (63.0 yerine 72.1 örnek/s). Sebep dinamik padding: batch büyüdükçe
batch içindeki en uzun dizi de uzuyor ve boşa giden padding artıyor — ortalama
padded uzunluk 16'da 105, 32'de 127 token.

Bu etki veriye bağlı, cihaza değil; T4'te de benzer davranması beklenir ama orada
ölçmedim. Yani bu hücre bir **deney**, hız garantisi değil.

Aşağıdaki değişiklik matematiksel olarak aynı etkin batch'i (32) korur, dolayısıyla
optimizer adım sayısı ve `warmup_steps` varsayımı bozulmaz. Denemek istersen çalıştır,
ilk birkaç yüz adımdaki `it/s` değerini not et ve kötüleşirse geri al.

In [ ]:
import re, yaml

PATH = 'src/configs/bert_hparams.yaml'

# Satir bazli degistiriyoruz: yaml.safe_dump ile geri yazmak dosyadaki tum
# aciklama satirlarini silerdi.
with open(PATH, encoding='utf-8') as f:
    lines = f.readlines()

DEGISIM = {'batch_size': 32, 'gradient_accumulation_steps': 1}
for i, line in enumerate(lines):
    for anahtar, deger in DEGISIM.items():
        # \s+ ile basliyor: sadece 'training:' altindaki girintili anahtarlar.
        # anahtar adinin tamamini esliyoruz, boylece eval_batch_size etkilenmez.
        if re.match(rf'^\s+{anahtar}:\s', line):
            girinti = re.match(r'^(\s+)', line).group(1)
            yorum = line.split('#', 1)[1].rstrip() if '#' in line else None
            lines[i] = f'{girinti}{anahtar}: {deger}' + (f'  #{yorum}\n' if yorum else '\n')

with open(PATH, 'w', encoding='utf-8') as f:
    f.writelines(lines)

cfg = yaml.safe_load(open(PATH, encoding='utf-8'))
bs = cfg['training']['batch_size']
accum = cfg['training']['gradient_accumulation_steps']
eff = bs * accum
print(f'batch_size={bs}  accum={accum}  -> etkin batch {eff}')
print(f"eval_batch_size={cfg['training']['eval_batch_size']} (degismemis olmali)")
print(f"optimizer adim/epoch: {263300 // eff:,} | warmup_steps: {cfg['training']['warmup_steps']}")
assert eff == 32, 'etkin batch 32 olmali - warmup_steps bu varsayima gore ayarli'

## 7. Eğit

Oturum koparsa **bu hücreyi tekrar çalıştır** — script `artifacts/bert_mlp_ckpt`
içindeki son checkpoint'i bulup kaldığı yerden devam eder (2. hücreyi
çalıştırdıysan Drive'da durur).

Sekmeyi açık tut; Colab boşta kalan oturumları kapatır.

In [ ]:
!python src/models/bert/bert_train.py

## 8. Hemen Hub'a yükle

Bunu eğitim biter bitmez yap. Oturum kapanırsa Drive'daki checkpoint kalsa bile
Hub'a yüklenmiş bir kopya en güvenlisidir — Space de oradan çekecek.

Token: huggingface.co/settings/tokens (write yetkisi).

In [ ]:
!ls -la artifacts/bert_mlp_ckpt/best_model/

from huggingface_hub import login, upload_folder, create_repo
login()  # write yetkili token

In [ ]:
REPO_ID = 'bariskiratt/duyguanalizi-berturk'  # kendi kullanici adinla degistir

create_repo(REPO_ID, repo_type='model', exist_ok=True)
upload_folder(
    repo_id=REPO_ID,
    folder_path='artifacts/bert_mlp_ckpt/best_model',
    commit_message='BERTurk + MLP 3 sinifli duygu modeli',
)
print(f'https://huggingface.co/{REPO_ID}')

## 9. Gerçek metrikleri üret

README'deki sayılar kayıp ağırlıklara ait; bu koşunun kendi metriklerini üret.

Çıktıda `✅ Base BERT model loaded (untrained)` satırını görürsen **dur** —
`bert_evaluate.py` özel modeli yükleyemeyip eğitilmemiş base BERT'e düşmüş
demektir ve ürettiği metrikler anlamsızdır.

In [ ]:
!python src/models/bert/bert_evaluate.py

## 10. Tahmini gözle doğrula

Sayılar iyi görünse de modelin makul cevap verdiğini kendi gözünle gör.

In [ ]:
import os
os.environ['SENTIMENT_CKPT'] = 'artifacts/bert_mlp_ckpt/best_model'

from src.inference.predictor import SentimentPredictor
p = SentimentPredictor()

ornekler = [
    'Ürün harika, tam beklediğim gibi. Çok memnun kaldım.',
    'Kargo geç geldi ve paket ezilmişti, hiç memnun kalmadım.',
    'Kargo hızlıydı ama ürün beklediğim gibi değil.',
    'Fiyatına göre idare eder.',
    'Bir haftadır kullanıyorum, şimdilik bir sorun yok.',
]
for r, t in zip(p.predict_batch(ornekler), ornekler):
    dagilim = '  '.join(f'{k}={v:.3f}' for k, v in r['scores'].items())
    print(f"{r['label']:<8} ({r['confidence']:.2f})  {t}\n         {dagilim}")

---

Buraya kadar geldiysen elinde Hub'da bir model var. Space kurulumu için
`DEPLOY.md` dosyasının 4. adımına geç.